# Atlas Iconocrático — Tutorial 3: Confiabilidade Inter-Codificadores e Integração Iconclass

**Tese de Doutorado** · PPGD/UFSC  
*Contrato Sexual Visual & Regime Iconocrático: Alegorias Femininas na Cultura Jurídica (1789–2024)*

---

## Sobre este tutorial

Este caderno cobre os **instrumentos de validação metodológica** da pesquisa:

1. **Confiabilidade inter-codificadores (Cohen's Kappa)** — como garantir que a codificação dos 10 indicadores é replicável e confiável entre dois pesquisadores independentes
2. **Integração Iconclass** — como conectar o sistema de codificação da tese com a biblioteca taxonômica internacional para humanidades digitais
3. **Matriz de Evidências** — como construir e consultar o banco de referências teóricas integrado ao corpus
4. **Fluxo completo de pesquisa** — do corpus à publicação

### Por que este tutorial importa para a banca?

A análise de confiabilidade inter-codificadores é **requisito metodológico obrigatório** em pesquisas de content analysis quantitativa (Krippendorff, 2004). Sem ela, os resultados do Tutorial 2 (testes H1–H5) ficam sem base de validade: não sabemos se dois pesquisadores distintos produziriam os mesmos valores de FEI, CII, WI, etc. para a mesma imagem.

O Kappa de Cohen é o padrão da área. Thresholds de Landis & Koch (1977):

| Kappa | Interpretação |
|-------|---------------|
| ≥ 0.80 | **Quase perfeita** — indicador pode ser mantido |
| 0.60–0.79 | **Moderada** — revisar definições operacionais |
| 0.40–0.59 | **Razoável** — redefinir critérios |
| < 0.40 | **Baixa** — eliminar ou reformular completamente |

---

> **Pré-requisito:** Tutorial 1 (corpus e codificação) — necessário para entender os 10 indicadores.  
> **Dependências:** `pandas`, `numpy`, `scikit-learn`


---
## 1. Confiabilidade Inter-Codificadores (Cohen's Kappa)

### 1.1 Por que a confiabilidade inter-codificadores é essencial?

Em análise de conteúdo quantitativa — o método que fundamenta os 10 indicadores deste corpus — a codificação depende do **julgamento do pesquisador**. Mesmo com definições operacionais precisas (como as do Tutorial 1), pesquisadores diferentes podem interpretar a mesma imagem de forma divergente.

**Exemplo:** diante da *Marianne* de Delacroix (*La Liberté guidant le peuple*, 1830), um codificador pode atribuir FEI=2 (corpo visível, seios perceptíveis sob as vestes) enquanto outro atribui FEI=3 (nudez parcial explícita). Sem um protocolo de confiabilidade, não sabemos qual critério prevalece — nem se os dados são comparáveis entre imagens codificadas em momentos diferentes.

A solução é simples: **dois codificadores independentes codificam as mesmas N imagens** e calculamos a concordância. O Cohen's Kappa desconta a concordância por acaso (ao contrário da simples percentagem de concordância), fornecendo uma medida mais honesta da confiabilidade real.

### Protocolo padrão da tese:
1. Selecionar amostra de **50 imagens aleatórias** do corpus (para o teste de confiabilidade)
2. Codificador 1 e Codificador 2 codificam independentemente, sem comunicação
3. Calcular Kappa por indicador
4. Indicadores com Kappa ≥ 0.80 → aprovados; 0.60–0.79 → reunião de consenso; < 0.60 → redefinir

### 1.2 Configuração e importações


In [ ]:
import sys
import os

# Adicionar diretório raiz ao path
toolkit_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if toolkit_root not in sys.path:
    sys.path.insert(0, toolkit_root)

# Imports do toolkit
from atlas_iconocratico.kappa import (
    calcular_kappa_indicador,
    calcular_kappa_todos,
    imprimir_relatorio_kappa,
    salvar_relatorio_kappa,
    identificar_discordancias,
    verificar_drift,
)
from atlas_iconocratico.iconclass_ import (
    listar_codigos_tese,
    buscar_por_texto,
    sugerir_iconclass_para_indicadores,
    exportar_mapeamento_tese,
    INDICADOR_ICONCLASS_MAP,
    CODIGOS_TESE,
)
from atlas_iconocratico.matriz import (
    GerenciadorMatriz,
    EntradaMatriz,
)

# Dependências
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Reprodutibilidade
np.random.seed(42)

print("✓ Dependências carregadas.")
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")


### 1.3 Simulação: dois codificadores, 50 imagens

Simulamos dois codificadores com **~80% de concordância** nas imagens — um nível realista que corresponde ao que ocorre *antes* da reunião de consenso, em uma primeira rodada de codificação.

A simulação usa o mesmo mecanismo do módulo `kappa.py`: o Codificador 2 replica os valores do Codificador 1 e, com 20% de probabilidade, atribui um valor aleatório diferente. Para indicadores com 4 categorias (0–3), uma discordância aleatória tende a afastar apenas 1 nível (e não 3), porque pesquisadores treinados raramente erram por 3 graus.


In [ ]:
# -------------------------------------------------------
# Parâmetros da simulação
# -------------------------------------------------------
N_IMAGENS = 50
IDS = [f"{i:04d}" for i in range(1, N_IMAGENS + 1)]

# Codificador 1 — codificação de base (realista, não aleatória pura)
dados_cod1 = {
    "Image_ID": IDS,
    # FEI: maioria em 1-2 (corporalidade sugerida/visível)
    "FEI":  np.random.choice([0, 1, 2, 3], N_IMAGENS, p=[0.15, 0.35, 0.35, 0.15]),
    # CII: concentrado em 2 (predominância clássica)
    "CII":  np.random.choice([0, 1, 2, 3], N_IMAGENS, p=[0.10, 0.25, 0.40, 0.25]),
    # PRI: categórico
    "PRI":  np.random.choice(["serena", "militante", "sacrificial"], N_IMAGENS, p=[0.50, 0.30, 0.20]),
    # SMI: distribuição moderada
    "SMI":  np.random.choice([0, 1, 2, 3], N_IMAGENS, p=[0.15, 0.35, 0.35, 0.15]),
    # SMS: categórico
    "SMS":  np.random.choice(["serena", "militante", "sacrificial"], N_IMAGENS, p=[0.55, 0.25, 0.20]),
    # AMCP: leve skew para baixo
    "AMCP": np.random.choice([0, 1, 2, 3], N_IMAGENS, p=[0.30, 0.35, 0.25, 0.10]),
    # MVI: maioria baixo
    "MVI":  np.random.choice([0, 1, 2, 3], N_IMAGENS, p=[0.40, 0.30, 0.20, 0.10]),
    # WI: concentrado em 2 (branquitude normativa)
    "WI":   np.random.choice([0, 1, 2, 3], N_IMAGENS, p=[0.05, 0.20, 0.45, 0.30]),
    # RI: maioria 1 (branqueamento)
    "RI":   np.random.choice([0, 1],        N_IMAGENS, p=[0.20, 0.80]),
    # AI: maioria 0 (alegoria pura)
    "AI":   np.random.choice([0, 1, 2, 3], N_IMAGENS, p=[0.45, 0.30, 0.15, 0.10]),
}

df_cod1 = pd.DataFrame(dados_cod1)

# -------------------------------------------------------
# Codificador 2: 80% concordância + discordâncias realistas
# -------------------------------------------------------
dados_cod2 = {k: v.copy() if isinstance(v, np.ndarray) else list(v)
              for k, v in dados_cod1.items()}

# Indicadores ordinais: discordância de ±1 nível (raramente mais)
for ind in ["FEI", "CII", "SMI", "AMCP", "MVI", "WI", "AI"]:
    arr = np.array(dados_cod1[ind], dtype=int)
    mask = np.random.random(N_IMAGENS) > 0.80   # 20% discordância
    ajuste = np.random.choice([-1, +1], size=mask.sum())
    arr[mask] = np.clip(arr[mask] + ajuste, 0, 3)
    dados_cod2[ind] = arr

# Indicadores categóricos: discordância de categoria inteira
for ind in ["PRI", "SMS"]:
    vals = list(dados_cod1[ind])
    for i in range(N_IMAGENS):
        if np.random.random() > 0.80:
            opcoes = [x for x in ["serena", "militante", "sacrificial"] if x != vals[i]]
            vals[i] = np.random.choice(opcoes)
    dados_cod2[ind] = vals

# RI (booleano): ~85% concordância (mais fácil — é 0/1)
ri_arr = np.array(dados_cod1["RI"], dtype=int)
mask_ri = np.random.random(N_IMAGENS) > 0.85
ri_arr[mask_ri] = 1 - ri_arr[mask_ri]
dados_cod2["RI"] = ri_arr

df_cod2 = pd.DataFrame(dados_cod2)

# Verificar concordâncias brutas por indicador
print("Concordâncias brutas (antes de calcular Kappa):")
print(f"{'Indicador':<8}  {'Conc. %':>8}  {'Discordâncias':>15}")
print("─" * 38)
for ind in ["FEI", "CII", "PRI", "SMI", "SMS", "AMCP", "MVI", "WI", "RI", "AI"]:
    conc = (df_cod1[ind] == df_cod2[ind]).sum()
    pct  = conc / N_IMAGENS * 100
    disc = N_IMAGENS - conc
    print(f"  {ind:<6}  {pct:>8.1f}%  {disc:>15}")


### 1.4 Salvamento e cálculo do Kappa

Salvamos os CSVs de cada codificador e executamos `calcular_kappa_todos()` — a função oficial do toolkit que calcula Cohen's Kappa para todos os indicadores, alinhando por `Image_ID`.

**Nota sobre o tipo de Kappa:**
- Indicadores **ordinais** (FEI, CII, SMI, AMCP, MVI, WI, AI): **weighted kappa linear** — penaliza discordâncias proporcionalmente à distância (uma discordância de 0→3 é pior que 0→1)
- Indicadores **categóricos** (PRI, SMS): **unweighted kappa** — qualquer discordância é igualmente penalizada (não há ordenação entre "serena", "militante", "sacrificial")
- RI (booleano): **unweighted kappa** — apenas dois estados possíveis


In [ ]:
# Salvar CSVs dos codificadores
os.makedirs("../data", exist_ok=True)
csv_cod1 = "../data/codificacao_COD1.csv"
csv_cod2 = "../data/codificacao_COD2.csv"
df_cod1.to_csv(csv_cod1, index=False, encoding="utf-8")
df_cod2.to_csv(csv_cod2, index=False, encoding="utf-8")
print(f"CSV Codificador 1: {csv_cod1}")
print(f"CSV Codificador 2: {csv_cod2}")
print(f"Imagens codificadas por cada: {N_IMAGENS}")


In [ ]:
# Calcular Kappa para todos os indicadores
resultados_kappa = calcular_kappa_todos(csv_cod1, csv_cod2)

# Imprimir relatório formatado
imprimir_relatorio_kappa(resultados_kappa)


In [ ]:
# Salvar relatório como CSV
csv_kappa = "../data/relatorio_kappa.csv"
salvar_relatorio_kappa(resultados_kappa, caminho=csv_kappa)


### 1.5 Interpretação dos valores de Kappa por indicador

A tabela abaixo conecta cada valor de Kappa à **decisão metodológica** correspondente:


In [ ]:
print("\n=== DECISÕES METODOLÓGICAS POR INDICADOR ===\n")

guia_indicadores = {
    "FEI": "Mais difícil: linha tênue entre 'corpo sugerido' (1) e 'corpo visível' (2)",
    "CII": "Razoável: referências clássicas são relatadas na bibliografia iconográfica",
    "PRI": "Difícil: 'militante' vs. 'sacrificial' exige consenso sobre intenção gestual",
    "SMI": "Mais fácil: contagem de atributos visíveis — objetivo",
    "SMS": "Difícil: igual ao PRI — sentido simbólico é interpretativo",
    "AMCP": "Moderado: presença de armas é objetiva; intensidade é subjetiva",
    "MVI": "Difícil: vulnerabilidade vs. maternidade — requer treinamento específico",
    "WI": "Moderado: traços raciais são interpretados culturalmente",
    "RI": "Mais fácil: binário 0/1 — menor margem de discordância",
    "AI": "Difícil: gradiente alegoria→biografia requer conhecimento histórico",
}

for ind, res in resultados_kappa["indicadores"].items():
    kappa = res["kappa"]
    cls = res["classificacao"]
    acao = res["acao"]
    nota = guia_indicadores.get(ind, "")

    if kappa >= 0.80:
        emoji = "✅"
    elif kappa >= 0.60:
        emoji = "⚠️ "
    elif kappa >= 0.40:
        emoji = "🔶"
    else:
        emoji = "❌"

    print(f"{emoji} {ind:<4}  κ = {kappa:.3f}  [{cls}]")
    print(f"       Ação: {acao}")
    print(f"       Nota: {nota}")
    print()

r = resultados_kappa["resumo"]
print(f"\n{'─'*60}")
print(f"Kappa médio:  {r['media_kappa']:.3f}")
print(f"Indicadores com κ ≥ 0.80: {r['n_excelente']}/10")
print(f"{'✅ CORPUS APROVADO' if r['todos_acima_080'] else '⚠️  REVISÃO NECESSÁRIA ANTES DA COLETA DEFINITIVA'}")


**Estratégia para indicadores abaixo de 0.80:**

1. **Reunião de consenso:** os dois codificadores revisam *juntos* cada imagem discordante, discutem os critérios e chegam a um acordo documentado
2. **Revisão das definições operacionais:** o protocolo de codificação (Apêndice A da tese) é refinado com exemplos adicionais e casos-limite explicitados
3. **Segunda rodada:** nova amostra de 20–30 imagens é codificada após a revisão; Kappa deve subir ≥ 0.80 antes de iniciar a codificação do corpus completo

### 1.6 Identificação de discordâncias

Após calcular o Kappa geral, é necessário **mapear as discordâncias específicas** para a reunião de consenso. A função `identificar_discordancias()` lista todas as imagens onde os dois codificadores divergiram em um indicador específico.


In [ ]:
# Identificar discordâncias no indicador mais difícil: FEI
print("=== Discordâncias em FEI (primeiras 10) ===\n")
disc_fei = identificar_discordancias(csv_cod1, csv_cod2, indicador="FEI")
if len(disc_fei) > 0:
    print(f"Total de discordâncias em FEI: {len(disc_fei)}")
    print(disc_fei.head(10).to_string(index=False))
    print(f"\nDistribuição das diferenças:")
    print(disc_fei["diferenca"].value_counts().to_string())
else:
    print("Nenhuma discordância encontrada em FEI.")


In [ ]:
# Identificar discordâncias em WI
print("\n=== Discordâncias em WI ===\n")
disc_wi = identificar_discordancias(csv_cod1, csv_cod2, indicador="WI")
if len(disc_wi) > 0:
    print(f"Total de discordâncias em WI: {len(disc_wi)}")
    print(disc_wi.to_string(index=False))
else:
    print("Nenhuma discordância encontrada em WI.")


In [ ]:
# Gerar tabela-resumo de discordâncias por indicador para a reunião de consenso
print("=== PAUTA DA REUNIÃO DE CONSENSO ===\n")
print(f"{'Indicador':<8}  {'Discordâncias':>15}  {'Imagens a revisar'}")
print("─" * 55)

total_para_revisar = set()
for ind in ["FEI", "CII", "PRI", "SMI", "SMS", "AMCP", "MVI", "WI", "RI", "AI"]:
    try:
        disc = identificar_discordancias(csv_cod1, csv_cod2, indicador=ind)
        n_disc = len(disc)
        ids = list(disc["Image_ID"]) if n_disc > 0 else []
        total_para_revisar.update(ids)
        preview = ", ".join(ids[:5]) + ("..." if len(ids) > 5 else "")
        print(f"  {ind:<6}  {n_disc:>15}  {preview}")
    except Exception as e:
        print(f"  {ind:<6}  {'─':>15}  ({e})")

print(f"\nTotal de imagens únicas para revisar: {len(total_para_revisar)}")
print(f"(Reunião de consenso estimada: ~{len(total_para_revisar) * 3} min a 3 min/imagem)")


### 1.7 Monitoramento de drift durante a codificação

O **drift de codificação** ocorre quando um pesquisador, ao longo de uma sessão longa de codificação, começa a aplicar os critérios de forma inconsistente com as primeiras codificações — um fenômeno de fadiga e deriva cognitiva documentado na literatura (Lombard et al., 2002).

A função `verificar_drift()` detecta mudanças sistemáticas na distribuição de um indicador ao longo da sequência de codificação, comparando janelas de N imagens com a baseline (primeiras N imagens).


In [ ]:
# Monitorar drift em FEI para o Codificador 1
print("=== MONITORAMENTO DE DRIFT — COD1 — FEI ===\n")
drift_fei = verificar_drift(csv_cod1, indicador="FEI", janela=20)

print(f"{'Janela':<12}  {'N':>5}  {'Média':>8}  {'DP':>8}  {'Drift':>10}  {'Alerta'}")
print("─" * 60)
for d in drift_fei:
    alerta = "⚠️  DRIFT!" if d.get("alerta_drift") else "—"
    drift_val = f"{d.get('drift_from_baseline', 0):.3f}" if "drift_from_baseline" in d else "baseline"
    print(f"  {d['janela']:<10}  {d.get('n', d.get('n', len(df_cod1))):>5}  "
          f"{d['media']:>8.3f}  {d['std']:>8.3f}  {drift_val:>10}  {alerta}")

print(f"\nInterpretação: drift > 0.5 pontos indica que o codificador está deslocando")
print(f"sistematicamente a escala — sinal de fadiga ou reinterpretação dos critérios.")


In [ ]:
# Drift em WI para o Codificador 1
print("=== MONITORAMENTO DE DRIFT — COD1 — WI ===\n")
drift_wi = verificar_drift(csv_cod1, indicador="WI", janela=20)

for d in drift_wi:
    alerta = "⚠️  DRIFT!" if d.get("alerta_drift") else "—"
    drift_val = f"{d.get('drift_from_baseline', 0):.3f}" if "drift_from_baseline" in d else "baseline"
    print(f"  Janela {d['janela']:<10}  média={d['media']:.3f}  std={d['std']:.3f}  "
          f"drift={drift_val}  {alerta}")


### 1.8 Protocolo de reunião de consenso

Quando indicadores ficam entre 0.60–0.79, o protocolo exige uma reunião estruturada:

**Formato:**
- Duração: 2–3 horas
- Participantes: Codificador 1, Codificador 2 e o pesquisador-autor como árbitro
- Material: projeção das imagens discordantes com os valores lado a lado

**Procedimento por imagem:**
1. Ambos os codificadores explicam *verbalmente* seu raciocínio
2. O árbitro identifica a fonte da discordância: definição ambígua? critério visual não especificado? especificidade cultural?
3. A decisão é tomada e registrada como **nota de protocolo**
4. A definição operacional no Apêndice A é atualizada se necessário

**Critério de aprovação:**
Após a reunião, reamostrar 20 imagens (10 das discordantes + 10 novas) e recalcular Kappa. Se ainda < 0.80 para algum indicador: reformular a definição ou considerar a eliminação do indicador do protocolo.

> **Registrar na metodologia da tese:** "A confiabilidade inter-codificadores foi avaliada em duas rodadas. Na primeira rodada (n=50), o Kappa médio foi κ=X.XX (min=X.XX, max=X.XX). Após reunião de consenso e revisão das definições operacionais de [indicadores], a segunda rodada (n=20) atingiu κ médio=X.XX, com todos os indicadores acima de 0.80."


---
## 2. Integração Iconclass

### 2.1 O que é Iconclass?

**Iconclass** é o sistema de classificação bibliográfico desenvolvido pelo Getty Research Institute e amplamente adotado em museus, bibliotecas e projetos de humanidades digitais para indexação de obras de arte. Cada conceito iconográfico recebe uma **notação alfanumérica hierárquica** que permite buscas precisas e interoperabilidade entre coleções.

**Por que integrar ao Atlas Iconocrático?**

1. **Interoperabilidade:** as imagens do corpus passam a ser pesquisáveis em sistemas internacionais (Rijksmuseum, British Museum, Bibliothèque nationale de France)
2. **Precisão vocabular:** substitui termos vagos ("figura feminina patriótica") por notações padronizadas ("44B3 — Pátria, personificação feminina da nação")
3. **Descoberta de novas imagens:** permite encontrar obras não mapeadas a partir de buscas por código
4. **Citabilidade:** a notação Iconclass é aceita como referência em artigos de história da arte e estudos visuais

**Estrutura hierárquica (exemplo):**
```
4       → Sociedade, Civilização, Cultura
 44     → Estado, Política
  44B   → Nação, Povo
   44B3 → Pátria — personificação feminina da nação
```

### 2.2 Códigos Iconclass mapeados para a tese


In [ ]:
# Listar todos os códigos mapeados para a tese
codigos = listar_codigos_tese()

print("=== MAPEAMENTO ICONCLASS DA TESE ===")
print(f"Categorias mapeadas: {len(codigos)}\n")

for categoria, mapa in codigos.items():
    print(f"  {'─'*50}")
    print(f"  {categoria.upper().replace('_', ' ')}")
    print(f"  {'─'*50}")
    for notacao, descricao in mapa.items():
        print(f"    {notacao:>12s}  →  {descricao}")
    print()


### 2.3 Busca por texto nos códigos mapeados

A função `buscar_por_texto()` permite encontrar os códigos Iconclass relevantes a partir de termos de busca em português. Útil durante a codificação para confirmar qual notação corresponde a um elemento visual específico.


In [ ]:
# Busca por diferentes termos temáticos
termos_busca = ["Justiça", "mulher", "Pátria", "balança", "espada"]

for termo in termos_busca:
    resultados = buscar_por_texto(termo)
    print(f"\n🔍 Busca por '{termo}' → {len(resultados)} resultado(s):")
    if resultados:
        for r in resultados:
            print(f"    {r['notacao']:>12s}  [{r['categoria']}]")
            print(f"                 {r['descricao']}")
    else:
        print(f"    (nenhum resultado nos códigos pré-mapeados)")


### 2.4 Sugestão automática de Iconclass para perfis de codificação

A função `sugerir_iconclass_para_indicadores()` recebe os valores dos indicadores de uma imagem e sugere automaticamente os códigos Iconclass mais relevantes. Isso acelera a anotação e garante consistência.

Demonstramos para **três perfis iconográficos arquetípicos** da tese:


In [ ]:
# Perfil 1: Marianne militante (ação bélica, corpo enfatizado)
perfil_marianne = {
    "Figure":  "Marianne",
    "FEI":  3,   # corpo enfatizado (nudez parcial — Delacroix 1830)
    "CII":  1,   # elementos clássicos mistos
    "SMI":  2,   # 2 atributos (bandeira + barricada)
    "AMCP": 2,   # armamento moderado (fuzil)
    "MVI":  1,   # vulnerabilidade leve (ferida)
    "AI":   0,   # alegoria pura
}

print("=" * 60)
print("PERFIL 1: Marianne Militante")
print("Exemplo: Delacroix, La Liberté guidant le peuple (1830)")
print("=" * 60)
print(f"Indicadores: {perfil_marianne}")
print()
sugestoes_m = sugerir_iconclass_para_indicadores(perfil_marianne)
print(f"Códigos Iconclass sugeridos ({len(sugestoes_m)}):")
for cod in sugestoes_m:
    for cat, mapa in CODIGOS_TESE.items():
        if cod in mapa:
            print(f"  {cod:>12s}  →  {mapa[cod]}")
            break
    else:
        print(f"  {cod:>12s}  →  (código externo ao mapeamento da tese)")


In [ ]:
# Perfil 2: Justitia serena (iconografia jurídica clássica)
perfil_justitia = {
    "Figure":  "Justitia",
    "FEI":  1,   # corpo sugerido (drapeado)
    "CII":  3,   # clássico puro (toga, perfil greco-romano)
    "SMI":  3,   # 4+ atributos (balança + espada + venda + livro)
    "AMCP": 1,   # espada como símbolo, não arma
    "MVI":  0,   # sem vulnerabilidade
    "AI":   0,   # alegoria pura por excelência
}

print("=" * 60)
print("PERFIL 2: Justitia Serena")
print("Exemplo: Justitia nas fachadas dos tribunais europeus")
print("=" * 60)
print(f"Indicadores: {perfil_justitia}")
print()
sugestoes_j = sugerir_iconclass_para_indicadores(perfil_justitia)
print(f"Códigos Iconclass sugeridos ({len(sugestoes_j)}):")
for cod in sugestoes_j:
    for cat, mapa in CODIGOS_TESE.items():
        if cod in mapa:
            print(f"  {cod:>12s}  →  {mapa[cod]}")
            break
    else:
        print(f"  {cod:>12s}  →  (ver hierarquia Iconclass completa)")


In [ ]:
# Perfil 3: República Branca (branqueamento normativo)
perfil_republica = {
    "Figure":  "República",
    "FEI":  1,   # corpo sugerido
    "CII":  2,   # predominância clássica
    "SMI":  1,   # 1 atributo (gorro frígio)
    "AMCP": 0,   # sem armamento
    "MVI":  0,   # sem vulnerabilidade
    "WI":   3,   # branquitude máxima
    "AI":   0,   # alegoria pura
}

print("=" * 60)
print("PERFIL 3: República Branca")
print("Exemplo: moedas e selos republicanos brasileiros pós-1889")
print("=" * 60)
print(f"Indicadores: {perfil_republica}")
print()
sugestoes_r = sugerir_iconclass_para_indicadores(perfil_republica)
print(f"Códigos Iconclass sugeridos ({len(sugestoes_r)}):")
for cod in sugestoes_r:
    for cat, mapa in CODIGOS_TESE.items():
        if cod in mapa:
            print(f"  {cod:>12s}  →  {mapa[cod]}")
            break
    else:
        print(f"  {cod:>12s}  →  (ver hierarquia)")


### 2.5 Mapeamento indicadores → Iconclass

O módulo contém um mapeamento sistemático de cada indicador da tese para os códigos Iconclass mais relevantes. Isso permite que um pesquisador que consulte o corpus saiba quais notações buscar para encontrar imagens de determinado perfil iconográfico.


In [ ]:
print("=== MAPEAMENTO INDICADORES → ICONCLASS ===\n")
for ind, info in INDICADOR_ICONCLASS_MAP.items():
    print(f"  {ind}  —  {info['descricao']}")
    print(f"       Códigos: {', '.join(info['codigos_relevantes'])}")
    print(f"       Nota:    {info['notas']}")
    print()


In [ ]:
# Exportar mapeamento completo como JSON
os.makedirs("../data", exist_ok=True)
exportar_mapeamento_tese(caminho="../data/iconclass_mapeamento_tese.json")

# Verificar arquivo gerado
with open("../data/iconclass_mapeamento_tese.json", encoding="utf-8") as f:
    mapeamento = json.load(f)

print(f"\nEstrutura do JSON exportado:")
for chave in mapeamento.keys():
    n = len(mapeamento[chave])
    print(f"  {chave}: {n} entradas")


---
## 3. Matriz de Evidências

### 3.1 Propósito e estrutura

A **Matriz de Evidências** é o banco de dados bibliográfico estruturado que integra as referências teóricas das Revisões Sistemáticas (SR1–SR9) com o corpus empírico codificado. Cada entrada na matriz registra:

1. **Referência ABNT completa** — citação padronizada
2. **Framework teórico** — filiação paradigmática (contrato_sexual, iconocracia, warburg, etc.)
3. **Claim principal** — argumento central relevante para a tese
4. **Força da evidência** — forte / moderada / fraca / **gap** (lacuna)
5. **País / Período / Figuras** — escopo empírico da referência
6. **Metodologia** — como o argumento foi construído
7. **Racialização** — se aborda branqueamento, RI, CII
8. **SR_source** — de qual busca sistemática veio

O campo **Evidence_strength = "gap"** é metodologicamente central: identifica explicitamente as **lacunas da literatura** que a tese se propõe a preencher — o que justifica a originalidade do projeto.

### 3.2 Criação e população da matriz


In [ ]:
# Criar gerenciador em diretório limpo (para demonstração)
os.makedirs("../data", exist_ok=True)
# Remover matriz anterior se existir (para limpar a demonstração)
import shutil
if Path("../data/matriz_evidencias.csv").exists():
    Path("../data/matriz_evidencias.csv").unlink()

gm = GerenciadorMatriz("../data")
print(f"GerenciadorMatriz criado em: {gm.base}")
print(f"Arquivo da matriz: {gm.csv_path}")


In [ ]:
# Adicionar referências reais do quadro teórico da tese
referencias = [
    # ─── SR1: Iconocracia e visualidade jurídica ───
    EntradaMatriz(
        Source_reference="MONDZAIN, Marie-José. Image, Icon, Economy: The Byzantine Origins of the Contemporary Imaginary. Stanford University Press, 2005.",
        Theoretical_framework="iconocracia",
        Main_claim="O regime iconocrático organiza a economia do visível — o Estado governa não apenas pela lei, mas pela imagem que produz de si mesmo",
        Evidence_strength="forte",
        Country="FR",
        Period="Século IX / teórico transtemporal",
        Key_figures_discussed="ícone cristão / alegoria estatal",
        Methodology="análise filosófica, história da arte",
        SR_source="SR1",
        Notes="Texto fundador do conceito central da tese — 'iconocracia'",
    ),
    EntradaMatriz(
        Source_reference="GOODRICH, Peter. Legal Emblems and the Art of Law: Obiter Depicta as the Vision of Governance. Cambridge University Press, 2014.",
        Theoretical_framework="iconocracia",
        Main_claim="Visiocracy: a lei se exerce como superfície visual; a iconografia forense é dispositivo de poder",
        Evidence_strength="forte",
        Country="UK",
        Key_figures_discussed="Justitia, Lady Justice",
        Support="monumento",
        Regime_link="normativo",
        Methodology="iconologia jurídica",
        SR_source="SR1",
    ),
    # ─── SR2: Contrato sexual (Pateman) ───
    EntradaMatriz(
        Source_reference="PATEMAN, Carole. The Sexual Contract. Stanford University Press, 1988.",
        Theoretical_framework="contrato_sexual",
        Main_claim="O contrato social pressupõe um contrato sexual que subordina mulheres — a fraternidade política é fundada na exclusão feminina da esfera pública",
        Evidence_strength="forte",
        Period="1688–presente (teórico)",
        Methodology="teoria política feminista",
        Racialisation="contrato sexual, gênero, corpo",
        SR_source="SR2",
        Notes="TEXTO FUNDACIONAL DA TESE — eixo central do quadro teórico",
    ),
    # ─── SR3: Alegorias nacionais ───
    EntradaMatriz(
        Source_reference="AGULHON, Maurice. Marianne au Combat: L'imagerie et la symbolique républicaines de 1789 à 1880. Flammarion, 1979.",
        Theoretical_framework="iconocracia",
        Main_claim="Institucionalização progressiva de Marianne como símbolo da República Francesa — da clandestinidade jacobina ao busto oficial",
        Evidence_strength="forte",
        Country="FR",
        Period="1789–1880",
        Key_figures_discussed="Marianne",
        Support="monumento, gravura, medalha",
        Regime_link="fundacional",
        Methodology="história da arte, iconologia",
        SR_source="SR3",
    ),
    EntradaMatriz(
        Source_reference="WARNER, Marina. Monuments and Maidens: The Allegory of the Female Form. Weidenfeld & Nicolson, 1985.",
        Theoretical_framework="feminismo",
        Main_claim="Alegorias femininas ocupam o espaço simbólico do Estado precisamente porque o Estado exclui as mulheres reais desse espaço — a alegoria substitui, não representa",
        Evidence_strength="forte",
        Country="FR, UK, USA",
        Key_figures_discussed="Marianne, Britannia, Liberty, Justitia, Columbia",
        Support="monumento, moeda, pintura",
        Methodology="análise cultural feminista",
        SR_source="SR3",
        Notes="Segunda referência fundacional — conecta Pateman com iconografia estatal",
    ),
    # ─── SR5: Racialização e branqueamento ───
    EntradaMatriz(
        Source_reference="MILLS, Charles W. The Racial Contract. Cornell University Press, 1997.",
        Theoretical_framework="contrato_racial",
        Main_claim="A supremacia branca é um sistema político cujo 'contrato racial' não escrito exclui pessoas de cor do status de pessoas morais e políticas plenas",
        Evidence_strength="forte",
        Racialisation="contrato racial, branqueamento sistêmico, CII, WI",
        SR_source="SR5",
        Notes="Fundamenta H2 (CII×WI) e H4 (RI=1 em BR e USA)",
    ),
    EntradaMatriz(
        Source_reference="SKIDMORE, Thomas E. Black into White: Race and Nationality in Brazilian Thought. Duke University Press, 1993.",
        Theoretical_framework="contrato_racial",
        Main_claim="A política de branqueamento brasileiro (1889–1930) foi deliberada e incluiu representações visuais do Estado",
        Evidence_strength="forte",
        Country="BR",
        Period="1889–1930",
        Racialisation="branqueamento, RI, política racial brasileira",
        SR_source="SR5",
        Notes="Suporte histórico para H4 no caso brasileiro",
    ),
    # ─── SR7: Método Warburg / Atlas ───
    EntradaMatriz(
        Source_reference="WARBURG, Aby. Bilderatlas Mnemosyne. Edição crítica: Bredekamp, Diers, Warnke (eds.). Akademie Verlag, 2020 [1929].",
        Theoretical_framework="warburg",
        Main_claim="Atlas como método iconológico: Pathosformel (fórmula de páthos), Nachleben (vida póstuma das imagens), Denkraum (espaço de pensamento entre imagens)",
        Evidence_strength="forte",
        Methodology="atlas/montagem, iconologia, Kulturwissenschaft",
        SR_source="SR7",
        Notes="Fundamenta o método Atlas Iconocrático e o conceito de 'painel'",
    ),
    EntradaMatriz(
        Source_reference="DIDI-HUBERMAN, Georges. Atlas ou le gai savoir inquiet. Minuit, 2011.",
        Theoretical_framework="warburg",
        Main_claim="Atualização do método warburgiano: o atlas como 'forma visual de saber' — montagem de imagens que pensa",
        Evidence_strength="forte",
        Methodology="atlas/montagem, filosofia da imagem",
        SR_source="SR7",
    ),
    # ─── LACUNA CENTRAL ───
    EntradaMatriz(
        Source_reference="(LACUNA IDENTIFICADA — SEM REFERÊNCIA EXISTENTE)",
        Theoretical_framework="contrato_sexual",
        Main_claim="Nenhum trabalho existente conecta sistematicamente: (1) teoria do contrato sexual de Pateman, (2) iconografia estatal comparativa multinacional, (3) codificação quantitativa de indicadores de gênero/raça, (4) método warburgiano de atlas",
        Evidence_strength="gap",
        Country="FR, UK, USA, DE, BR",
        Period="1789–2024",
        Methodology="ausência de método integrado",
        SR_source="SR2",
        Notes="LACUNA CENTRAL DA TESE — confirma originalidade e justifica o projeto",
    ),
]

n = gm.adicionar_lote(referencias)
print(f"\n{n} entradas adicionadas à matriz.")
print(f"Total na matriz: {gm.contar_entradas()}")


### 3.3 Análise de lacunas

A análise de lacunas é o argumento de **originalidade da tese**: demonstra que nenhum trabalho existente integra os quatro elementos teórico-metodológicos que o Atlas Iconocrático combina.


In [ ]:
# Imprimir relatório completo de lacunas
gm.imprimir_relatorio_lacunas()


In [ ]:
# Acessar e formatar o relatório programaticamente
relatorio = gm.relatorio_lacunas()

print("\n=== LACUNA CENTRAL — ANÁLISE DE ORIGINALIDADE ===\n")
lc = relatorio["lacuna_central"]
print(f"  Referências Pateman / contrato sexual:  {lc['refs_pateman']}")
print(f"  Referências iconocracia:                {lc['refs_iconocracia']}")
print(f"  Conexão sistêmica Pateman + visual:     {lc['conexao_pateman_visual']}")

print(f"\n  Diagnóstico: {lc['interpretacao']}")

print(f"\n  → Distribuição por força de evidência:")
for forca, n in relatorio["por_forca"].items():
    barra = "█" * n
    print(f"     {forca:<10}  {n:>2d}  {barra}")

print(f"\n  → Cobertura por país:")
for pais, n in relatorio["por_pais"].items():
    print(f"     {pais}  {n:>2d} referências")


### 3.4 Exportação de bibliografia ABNT

A função `exportar_bibliografia_abnt()` gera um arquivo de texto com todas as referências únicas da matriz no formato ABNT — pronto para ser copiado para a seção de referências da tese.


In [ ]:
# Exportar bibliografia ABNT
gm.exportar_bibliografia_abnt(caminho="../data/bibliografia_abnt.txt")

# Exibir o conteúdo exportado
with open("../data/bibliografia_abnt.txt", encoding="utf-8") as f:
    print(f.read())


In [ ]:
# Exportar matriz completa em JSON (para arquivamento e publicação)
caminho_json = gm.exportar_matriz_completa(formato="json")
print(f"\nMatriz exportada como JSON: {caminho_json}")

# Exportar como CSV
caminho_csv_m = gm.exportar_matriz_completa(formato="csv")
print(f"Matriz exportada como CSV:  {caminho_csv_m}")


### 3.5 Integração corpus + matriz

A função `integrar_corpus_e_matriz()` une os três datasets principais em um único arquivo para análise cruzada: metadados do corpus visual, codificações dos 10 indicadores e referências teóricas da matriz.

> **Nota:** Na tese real, este dataset integrado será a base para as visualizações do Atlas (painéis I–VIII) que correlacionam achados teóricos com evidências empíricas de imagens específicas.


In [ ]:
from atlas_iconocratico.matriz import integrar_corpus_e_matriz

# Verificar arquivos disponíveis para integração
arquivos_para_integrar = {
    "corpus_visual":  "../data/corpus_visual_master.csv",
    "codificacao":    "../data/corpus_simulado_nb02.csv",  # gerado no NB02
    "matriz":         "../data/matriz_evidencias.csv",
}

print("Arquivos disponíveis para integração:")
for nome, caminho in arquivos_para_integrar.items():
    existe = Path(caminho).exists()
    status = "✅" if existe else "❌ não encontrado"
    print(f"  {status}  {nome:<20}  {caminho}")

# Integrar codificação + corpus
resultado = integrar_corpus_e_matriz(
    corpus_csv   = arquivos_para_integrar["corpus_visual"],
    codificacao_csv = arquivos_para_integrar["codificacao"],
    output       = "../data/dataset_integrado.csv",
)
if resultado:
    df_int = pd.read_csv(resultado)
    print(f"\nDataset integrado: {df_int.shape[0]} linhas × {df_int.shape[1]} colunas")
    print("Colunas disponíveis:")
    print(list(df_int.columns))


---
## 4. Fluxo Completo de Pesquisa

Este diagrama conecta todos os módulos e notebooks do Atlas Iconocrático em uma sequência metodológica coerente:

```
┌─────────────────────────────────────────────────────────────────┐
│                    ATLAS ICONOCRÁTICO                           │
│           Fluxo Completo de Pesquisa (1789–2024)               │
└─────────────────────────────────────────────────────────────────┘

FASE 1 — CORPUS (Tutorial 1: corpus.py)
  ┌──────────────────────────────────────────────┐
  │ 1. Seleção das imagens (300–500)             │
  │    Critérios: suporte oficial, figura        │
  │    feminina alegórica, 5 países              │
  │    Output: corpus_visual_master.csv          │
  └──────────────────────────────────────────────┘
                       │
                       ▼

FASE 2 — CODIFICAÇÃO (Tutorial 1: codificacao.py)
  ┌──────────────────────────────────────────────┐
  │ 2. Aplicação dos 10 indicadores              │
  │    Por codificador treinado, imagem por      │
  │    imagem, com o formulário digital          │
  │    Output: codificacao_COD1.csv              │
  │            codificacao_COD2.csv              │
  └──────────────────────────────────────────────┘
                       │
                       ▼

FASE 3 — CONFIABILIDADE (Tutorial 3: kappa.py)  ◄── ESTE TUTORIAL
  ┌──────────────────────────────────────────────┐
  │ 3. Cálculo de Cohen's Kappa por indicador    │
  │    Reunião de consenso para κ < 0.80         │
  │    Segunda rodada até κ ≥ 0.80 em todos      │
  │    Output: relatorio_kappa.csv               │
  └──────────────────────────────────────────────┘
                       │
                       ▼

FASE 4 — ICONCLASS (Tutorial 3: iconclass_.py)  ◄── ESTE TUTORIAL
  ┌──────────────────────────────────────────────┐
  │ 4. Anotação Iconclass das imagens            │
  │    sugerir_iconclass_para_indicadores()      │
  │    Output: anotacoes_iconclass.csv           │
  │            iconclass_mapeamento_tese.json    │
  └──────────────────────────────────────────────┘
                       │
                       ▼

FASE 5 — ESTATÍSTICA (Tutorial 2: estatistica.py)
  ┌──────────────────────────────────────────────┐
  │ 5. Descritivas + Testes H1–H5                │
  │    Visualizações para a tese                 │
  │    Output: output/graficos/ e tabelas/       │
  │            resultados_hipoteses.txt          │
  └──────────────────────────────────────────────┘
                       │
                       ▼

FASE 6 — MATRIZ DE EVIDÊNCIAS (Tutorial 3: matriz.py)
  ┌──────────────────────────────────────────────┐
  │ 6. Integração referências teóricas + corpus  │
  │    Análise de lacunas → originalidade        │
  │    Output: matriz_evidencias.csv             │
  │            bibliografia_abnt.txt             │
  └──────────────────────────────────────────────┘
                       │
                       ▼

FASE 7 — ATLAS (Apêndice B da tese)
  ┌──────────────────────────────────────────────┐
  │ 7. Painéis I–VIII: montagem warburgiana      │
  │    das imagens com evidências estatísticas   │
  │    e anotações Iconclass                     │
  │    Output: atlas_iconocratico.pdf            │
  └──────────────────────────────────────────────┘
```

---

### O que fazer a seguir: da simulação ao corpus real

| Etapa | Ação concreta | Prazo estimado |
|-------|---------------|----------------|
| **Coleta** | Selecionar 300–500 imagens reais com proveniência documentada (museus, arquivos, Numista, coleções institucionais) | 3–4 semanas |
| **Treinamento** | Treinar Codificador 2 com o protocolo do Apêndice A, codificar 20 imagens-piloto em conjunto | 1 semana |
| **Codificação** | Cada codificador codifica independentemente as 300–500 imagens | 4–6 semanas |
| **Kappa Round 1** | Executar `calcular_kappa_todos()`, identificar discordâncias | 1 dia |
| **Consenso** | Reunião de consenso para indicadores com κ < 0.80 | 1–2 dias |
| **Kappa Round 2** | Recalcular κ em nova amostra de 20 imagens | 1 dia |
| **Iconclass** | Anotar corpus com códigos Iconclass usando `sugerir_iconclass_para_indicadores()` | 1–2 semanas |
| **Análise** | Executar Tutorial 2 com dados reais | 1 dia |
| **Atlas** | Montar painéis warburgians com base nos dados | 2–3 semanas |

> **Critério de qualidade mínimo para a defesa:** κ ≥ 0.80 em todos os 10 indicadores, na segunda rodada, com n ≥ 50 imagens de teste. Registrar os valores exatos no Apêndice Metodológico.

---

### Recursos adicionais

- **Iconclass online:** https://iconclass.org — busca e navegação na hierarquia completa
- **Protocolo de codificação:** Apêndice A da tese (gerado pelo `apendice_metodologico.docx`)
- **Documentação do toolkit:** `README.md` na raiz do repositório
- **Módulos do Atlas:** `atlas_iconocratico/` — corpus, codificacao, estatistica, kappa, iconclass_, matriz


In [ ]:
# Verificar todos os arquivos gerados por este notebook
print("=== ARQUIVOS GERADOS PELO TUTORIAL 3 ===\n")
arquivos = [
    ("../data/codificacao_COD1.csv",              "Codificação do Codificador 1 (50 imagens)"),
    ("../data/codificacao_COD2.csv",              "Codificação do Codificador 2 (50 imagens)"),
    ("../data/relatorio_kappa.csv",               "Relatório Kappa por indicador"),
    ("../data/iconclass_mapeamento_tese.json",    "Mapeamento Iconclass completo (JSON)"),
    ("../data/matriz_evidencias.csv",             "Matriz de evidências (referências teóricas)"),
    ("../data/matriz_evidencias_completa.csv",    "Matriz exportada (CSV completo)"),
    ("../data/matriz_evidencias_completa.json",   "Matriz exportada (JSON)"),
    ("../data/bibliografia_abnt.txt",             "Bibliografia em formato ABNT"),
    ("../data/dataset_integrado.csv",             "Dataset integrado corpus + codificação"),
]
for caminho, descricao in arquivos:
    existe = Path(caminho).exists()
    status = "✅" if existe else "—"
    size = ""
    if existe:
        size_b = Path(caminho).stat().st_size
        size = f"  {size_b/1024:.1f} KB" if size_b > 1024 else f"  {size_b} B"
    print(f"  {status}  {descricao:<45}  {caminho}{size}")
